In [1]:
"""
Bidirectional RNN From Scratch using NumPy
==========================================

A Bidirectional RNN processes a sequence in two directions:

1. Forward pass:
      left  -> right

2. Backward pass:
      right -> left

At each timestep:

    context_t = [h_forward_t ; h_backward_t]

This allows every position to contain:
    - past information
    - future information

Final context shape:
    (SEQ_LEN, 2 * HIDDEN_DIM)
"""

import numpy as np

# ---------------------------------------------------
# Random Generator
# ---------------------------------------------------
rng = np.random.default_rng(42)

# ---------------------------------------------------
# Hyperparameters
# ---------------------------------------------------
SEQ_LEN = 6
INPUT_DIM = 3
HIDDEN_DIM = 4

# ---------------------------------------------------
# Initialize Weights
# ---------------------------------------------------
def init_rnn_weights():

    scale = 0.3

    params = {

        # Forward Direction
        "W_xh_fwd": rng.normal(
            scale=scale,
            size=(INPUT_DIM, HIDDEN_DIM)
        ),

        "W_hh_fwd": rng.normal(
            scale=scale,
            size=(HIDDEN_DIM, HIDDEN_DIM)
        ),

        "b_h_fwd": np.zeros(HIDDEN_DIM),

        # Backward Direction
        "W_xh_bwd": rng.normal(
            scale=scale,
            size=(INPUT_DIM, HIDDEN_DIM)
        ),

        "W_hh_bwd": rng.normal(
            scale=scale,
            size=(HIDDEN_DIM, HIDDEN_DIM)
        ),

        "b_h_bwd": np.zeros(HIDDEN_DIM),
    }

    return params

# ---------------------------------------------------
# Single RNN Pass
# ---------------------------------------------------
def rnn_pass(sequence, W_xh, W_hh, b_h):

    """
    Runs a vanilla RNN over a sequence.

    Parameters
    ----------
    sequence : (T, INPUT_DIM)

    Returns
    -------
    outputs : (T, HIDDEN_DIM)
    """

    T = sequence.shape[0]

    # Initial hidden state
    h = np.zeros(HIDDEN_DIM)

    outputs = []

    for t in range(T):

        x_t = sequence[t]

        # Vanilla RNN Equation
        h = np.tanh(
            x_t @ W_xh +
            h @ W_hh +
            b_h
        )

        outputs.append(h.copy())

    return np.array(outputs)

# ---------------------------------------------------
# Bidirectional RNN
# ---------------------------------------------------
def bidirectional_rnn(sequence, params):

    """
    Run forward and backward RNN passes.

    Returns
    -------
    context : (T, 2 * HIDDEN_DIM)
    """

    # -----------------------------
    # Forward Pass
    # -----------------------------
    h_fwd = rnn_pass(
        sequence,
        params["W_xh_fwd"],
        params["W_hh_fwd"],
        params["b_h_fwd"]
    )

    # -----------------------------
    # Backward Pass
    # -----------------------------
    reversed_sequence = sequence[::-1]

    h_bwd_reversed = rnn_pass(
        reversed_sequence,
        params["W_xh_bwd"],
        params["W_hh_bwd"],
        params["b_h_bwd"]
    )

    # Reverse backward outputs
    # so time indices align
    h_bwd = h_bwd_reversed[::-1]

    # -----------------------------
    # Concatenate
    # -----------------------------
    context = np.concatenate(
        [h_fwd, h_bwd],
        axis=1
    )

    return context, h_fwd, h_bwd

# ---------------------------------------------------
# Main
# ---------------------------------------------------
if __name__ == "__main__":

    # Random input sequence
    sequence = rng.normal(
        size=(SEQ_LEN, INPUT_DIM)
    )

    # Initialize parameters
    params = init_rnn_weights()

    # Run BiRNN
    context, h_fwd, h_bwd = bidirectional_rnn(
        sequence,
        params
    )

    # ------------------------------------------------
    # Outputs
    # ------------------------------------------------
    print("\nInput Shape")
    print(sequence.shape)

    print("\nForward Hidden Shape")
    print(h_fwd.shape)

    print("\nBackward Hidden Shape")
    print(h_bwd.shape)

    print("\nContext Shape")
    print(context.shape)

    print("\nExpected Context Shape")
    print((SEQ_LEN, 2 * HIDDEN_DIM))

    print("\nFirst Context Vector")
    print(np.round(context[0], 3))

    print("\nForward Hidden State at t=0")
    print(np.round(h_fwd[0], 3))

    print("\nBackward Hidden State at t=0")
    print(np.round(h_bwd[0], 3))


Input Shape
(6, 3)

Forward Hidden Shape
(6, 4)

Backward Hidden Shape
(6, 4)

Context Shape
(6, 8)

Expected Context Shape
(6, 8)

First Context Vector
[-0.179  0.125  0.207  0.144 -0.107 -0.271  0.57   0.284]

Forward Hidden State at t=0
[-0.179  0.125  0.207  0.144]

Backward Hidden State at t=0
[-0.107 -0.271  0.57   0.284]
